# 05 — Semissupervisionado e importância

## Objetivo

Self-training melhora o ranking quando parte dos rótulos é ocultada? E como
interpretar globalmente o modelo escolhido?

A demonstração semissupervisionada é curta. Depois usamos importâncias nativas
e Permutation Importance, sem explicações individuais.

In [ ]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import json
import time
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.semi_supervised import SelfTrainingClassifier

from src.auxiliares import (
    COLUNAS_NOMINAIS,
    PARAMETROS_REFERENCIA,
    avaliar_probabilidades,
    carregar_base_preparada,
    criar_modelo_gradiente,
    separar_dados,
)
from src.visual_utils import grafico_importancia_variaveis

dados = carregar_base_preparada(RAIZ)
X_treino, X_validacao, X_teste, y_treino, y_validacao, y_teste = separar_dados(dados)
caminho_parametros = RAIZ / "models" / "parametros_gradient_boosting.json"
parametros = json.loads(caminho_parametros.read_text()) if caminho_parametros.exists() else PARAMETROS_REFERENCIA.copy()

## Como simular poucos rótulos disponíveis?

## Qual estimador será usado no self-training?

In [ ]:
indices_nominais = [X_treino.columns.get_loc(coluna) for coluna in COLUNAS_NOMINAIS]
indices_numericos = [i for i in range(X_treino.shape[1]) if i not in indices_nominais]

def criar_estimador_base():
    preprocessamento = ColumnTransformer([
        ("nominais", OneHotEncoder(handle_unknown="ignore", sparse_output=False), indices_nominais),
        ("numericas", StandardScaler(), indices_numericos),
    ])
    return Pipeline([
        ("preprocessamento", preprocessamento),
        ("modelo", LogisticRegression(max_iter=2000, random_state=42)),
    ])

## O self-training supera o mesmo subconjunto rotulado?

O self-training demonstra a técnica, mas não melhora a AP / PR-AUC neste
experimento. Ele não será a solução final.

## Quais variáveis o modelo usa globalmente?

In [ ]:
fig = grafico_importancia_variaveis(
    importancias_nativas,
    titulo="Importância nativa do Gradient Boosting",
)
fig.show()

## A importância permanece ao embaralhar cada feature original?

In [ ]:
fig = grafico_importancia_variaveis(
    importancias_permutacao,
    coluna_valor="importancia_media",
    titulo="Permutation Importance na validação",
    coluna_desvio="desvio",
)
fig.show()

## Resultado

O status de pagamento mais recente lidera as duas leituras. Importância
preditiva não implica causalidade, e variáveis mensais correlacionadas podem
dividir importância.